# HW13 — Токенизация, инференс BERT, fine-tuning для классификации текста

Домашнее задание к семинару 13: полный pipeline на датасете [`emotion`](https://huggingface.co/datasets/emotion) (6 классов эмоций в коротких английских текстах).

**План:** импорты и seed → данные → токенизация → инференс готовой pretrained-модели (другая постановка) → fine-tuning `DistilBERT` → метрики на `test`, confusion matrix, примеры → `artifacts/`.

**Рабочая директория:** при **Run All** Jupyter должен использовать папку `homeworks/HW13` (относительные пути `./artifacts/` и чекпоинты `artifacts/trainer_output/`).


## 2.3.1. Импорты, seed и среда


In [1]:
# Зависимости HF (если уже стоят — pip быстро отработает)
%pip install -q "datasets>=2.14" "transformers>=4.40" "accelerate>=0.26"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn  # noqa: F401 — требование ДЗ: sklearn подключён
import torch

from datasets import load_dataset
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.metrics import ConfusionMatrixDisplay
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)


In [3]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


SEED = 42
set_seed(SEED)

import datasets
import transformers

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"torch: {torch.__version__}")
print(f"datasets: {datasets.__version__}")
print(f"transformers: {transformers.__version__}")


Device: cpu
torch: 2.10.0+cpu
datasets: 4.8.4
transformers: 5.5.0


## 2.3.2. Данные и первичный анализ

Датасет **`emotion`**: классификация **эмоции** по короткому английскому тексту. Официальные сплиты HuggingFace: `train` / `validation` / `test`. Validation — для выбора лучшего чекпоинта по метрике; **test** используется **один раз** в конце.


In [4]:
dataset = load_dataset("emotion")
label_names = dataset["train"].features["label"].names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in id2label.items()}
num_labels = len(label_names)

train_ds = dataset["train"]
val_ds = dataset["validation"]
test_ds = dataset["test"]

print("Размеры:")
print("  train:", f"{len(train_ds):,}")
print("  validation:", f"{len(val_ds):,}")
print("  test:", f"{len(test_ds):,}")
print()
print("Классы (" + str(num_labels) + "):", label_names)

show_idx = [0, 1, 2, len(train_ds) // 2, len(train_ds) - 1]
rows = []
for i in show_idx:
    ex = train_ds[i]
    rows.append(
        {"text": ex["text"], "label_id": ex["label"], "label_name": label_names[ex["label"]]}
    )
display(pd.DataFrame(rows))


Размеры:
  train: 16,000
  validation: 2,000
  test: 2,000

Классы (6): ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


,text,label_id,label_name
0,i didnt feel humiliated,0,sadness
1,i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake,0,sadness
2,im grabbing a minute to post i feel greedy wrong,3,anger
3,i feel cool reading this book especially when i take it along to read while waiting for a doctors appointment,1,joy
4,i know a lot but i feel so stupid because i can not portray it,0,sadness


**Что классифицируем:** по фразе выбрать одну из шести эмоций. Тексты короткие; набор относительно сбалансирован — типичный учебный сценарий для sequence classification.


## 2.3.3. Токенизация

Показываем токены, `input_ids`, `attention_mask`, special tokens; затем батч с **padding** и **truncation** (`max_length=32` для наглядности).


In [5]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Special tokens:", tokenizer.special_tokens_map)
print("[CLS] id:", tokenizer.cls_token_id, "[SEP] id:", tokenizer.sep_token_id, "PAD id:", tokenizer.pad_token_id)

sample_texts = train_ds[:5]["text"]
for t in sample_texts:
    enc = tokenizer(t, add_special_tokens=True, return_tensors=None)
    print()
    print("TEXT:", t[:120] + ("..." if len(t) > 120 else ""))
    print("tokens:", tokenizer.convert_ids_to_tokens(enc["input_ids"]))
    print("input_ids:", enc["input_ids"])
    print("attention_mask:", enc["attention_mask"])


Special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
[CLS] id: 101 [SEP] id: 102 PAD id: 0

TEXT: i didnt feel humiliated
tokens: ['[CLS]', 'i', 'didn', '##t', 'feel', 'humiliated', '[SEP]']
input_ids: [101, 1045, 2134, 2102, 2514, 26608, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1]

TEXT: i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
tokens: ['[CLS]', 'i', 'can', 'go', 'from', 'feeling', 'so', 'hopeless', 'to', 'so', 'damned', 'hopeful', 'just', 'from', 'being', 'around', 'someone', 'who', 'cares', 'and', 'is', 'awake', '[SEP]']
input_ids: [101, 1045, 2064, 2175, 2013, 3110, 2061, 20625, 2000, 2061, 9636, 17772, 2074, 2013, 2108, 2105, 2619, 2040, 14977, 1998, 2003, 8300, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

TEXT: im grabbing a minute to post i feel greedy wrong
tokens: ['[CLS]', 'im', 'grabbing'

In [6]:
batch = train_ds[:4]["text"]
batch_enc = tokenizer(
    batch,
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors="pt",
)
print("max_length=32, padding=True, truncation=True")
print("input_ids shape:", tuple(batch_enc["input_ids"].shape))
print("attention_mask shape:", tuple(batch_enc["attention_mask"].shape))
print("attention_mask последней строки:", batch_enc["attention_mask"][-1].tolist())


max_length=32, padding=True, truncation=True
input_ids shape: (4, 23)
attention_mask shape: (4, 23)
attention_mask последней строки: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]


## 2.3.4. Инференс готовой pretrained модели

Модель **`distilbert-base-uncased-finetuned-sst-2-english`** — бинарный **sentiment** (POSITIVE/NEGATIVE), не 6 эмоций. На наших примерах видно осмысленную бинарную разметку, но **пространство меток не совпадает** с `emotion`; для итоговой задачи нужен fine-tuning своей головы (ниже).


In [7]:
pretrained_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    tokenizer="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
)

demo_texts = [
    "I am so happy and grateful today!",
    "This makes me furious and disappointed.",
    "I feel a bit sad and lonely.",
    "What a wonderful surprise party!",
    "I am scared about tomorrow's exam.",
]
for txt in demo_texts:
    out = pretrained_pipe(txt)[0]
    print(out["label"], f"({out['score']:.3f})", "|", txt)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

POSITIVE (1.000) | I am so happy and grateful today!
NEGATIVE (1.000) | This makes me furious and disappointed.
NEGATIVE (0.999) | I feel a bit sad and lonely.
POSITIVE (1.000) | What a wonderful surprise party!
NEGATIVE (0.999) | I am scared about tomorrow's exam.


**Итог:** готовый инференс полезен как демонстрация логитов/уверенности и домена SST-2, но **не решает** постановку `emotion` без дообучения.


## 2.3.5. Fine-tuning

`DistilBERT` + классификационная голова на 6 меток. Токенизация батчами (`max_length=128`). Лучший чекпоинт по **`eval_f1_macro` на validation** (`load_best_model_at_end`).


In [8]:
def tokenize_batch(examples):
    out = tokenizer(examples["text"], truncation=True, max_length=128)
    out["labels"] = examples["label"]
    return out


tok_train = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
tok_val = val_ds.map(tokenize_batch, batched=True, remove_columns=val_ds.column_names)
tok_test = test_ds.map(tokenize_batch, batched=True, remove_columns=test_ds.column_names)

for ds_ in (tok_train, tok_val, tok_test):
    ds_.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1_macro": float(f1_score(labels, preds, average="macro")),
    }


OUT_DIR = Path("artifacts") / "trainer_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    seed=SEED,
    report_to="none",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\Гошанский\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.208656,0.192156,0.930500,0.904444
2,0.124766,0.157587,0.932500,0.905901
3,0.085542,0.147754,0.944000,0.919183


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Гошанский\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Гошанский\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3000, training_loss=0.23987743894259134, metrics={'train_runtime': 3654.08, 'train_samples_per_second': 13.136, 'train_steps_per_second': 0.821, 'total_flos': 584777647046016.0, 'train_loss': 0.23987743894259134, 'epoch': 3.0})

## 2.3.6. Метрики на test, confusion matrix, примеры и артефакты

Минимум: **accuracy**, **f1_macro**, матрица ошибок на **test**, 5–10 примеров в выводе; сохранение `./artifacts/sample_predictions.csv` и `./artifacts/confusion_matrix.png`.


In [ ]:
# Не вызываем trainer.evaluate(test): в Jupyter NotebookProgressCallback даёт
# RuntimeError: on_train_begin must be called before on_evaluate.
# Один predict + compute_metrics — та же финальная оценка на test.
test_pred = trainer.predict(tok_test)
logits = test_pred.predictions
y_true = test_pred.label_ids
y_pred = np.argmax(logits, axis=-1)

test_metrics = compute_metrics((logits, y_true))
print("=== Финальная оценка на test (один раз) ===")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"  eval_{k}: {v:.4f}")
    else:
        print(f"  eval_{k}: {v}")

test_acc = float(test_metrics["accuracy"])
test_f1m = float(test_metrics["f1_macro"])
print()
print("accuracy (test):", f"{test_acc:.4f}")
print("f1_macro (test):", f"{test_f1m:.4f}")

cm = confusion_matrix(y_true, y_pred, labels=list(range(num_labels)))
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=True)
ax.set_title("Confusion matrix (test)")
plt.tight_layout()
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)
cm_path = ARTIFACTS / "confusion_matrix.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", cm_path.resolve())


In [ ]:
probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
confidences = probs.max(axis=1)

texts_test = test_ds["text"]
true_names = [label_names[i] for i in y_true]
pred_names = [label_names[i] for i in y_pred]

df_show = pd.DataFrame(
    {
        "text": texts_test,
        "true_label": true_names,
        "pred_label": pred_names,
        "confidence": confidences,
    }
)

print(df_show.head(10).to_string(index=False))

sample_path = ARTIFACTS / "sample_predictions.csv"
df_show.head(200).to_csv(sample_path, index=False)
print()
print("Saved:", sample_path.resolve(), "(первые 200 строк test)")


In [ ]:
wrong = df_show[df_show["true_label"] != df_show["pred_label"]].copy()
wrong = wrong.sort_values("confidence", ascending=False)
print("Ошибки с высокой уверенностью:")
print(wrong.head(5).to_string(index=False))
print()
print("Пограничные (низкая уверенность):")
print(df_show.nsmallest(5, "confidence").to_string(index=False))


In [ ]:
print("--- Для вставки в report.md ---")
print("test_accuracy:", f"{test_acc:.4f}")
print("test_f1_macro:", f"{test_f1m:.4f}")
print("device:", device)
